# 1 - Bibliotecas

In [5]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, date

from pyspark.sql.functions import col, count, when, isnan, countDistinct, approx_count_distinct, concat, col, lit, substring
from pyspark.sql.types import DoubleType, StringType, NumericType
from pyspark.sql.types import *
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

#

# 2 - Carregando os dados

In [6]:
# Pastas
pastas_alvo = [
    "base_dados_cadastrais",
    "base_score_bureau_movel",
    "base_score_bureau_movel_full",
    "base_telco",
    "book_pagamento/dados_pagamento",
    "book_pagamento",
    "bases_recarga"
]

In [7]:
# --------------------------
# Endereços: Adaptar Bucket
# --------------------------

base_drive = "/content/gdrive/MyDrive/Hackathon_POD/Dados"
base_local = "/content/dados_locais"

In [8]:
# -----------------------------------------
# Carregando os dados para a memoria local
# -----------------------------------------

for pasta in pastas_alvo:
  origem = os.path.join(base_drive, pasta)
  destino = os.path.join(base_local, pasta)

  if os.path.exists(origem):
    print(f'Procecando pasta {pasta}..')

    os.makedirs(destino, exist_ok = True)

    !cp -rn '{origem}/.' '{destino}'
    print(f'Sucesso, Copiado para: {destino}')
  else:
    print(f'ERRO: Pasta não encontrada no Drive: {origem}')

print('\n--PROCESSO FINALIZADO--')
print(f'Pastas criadas localmente: {os.listdir(base_local)}')

Procecando pasta base_dados_cadastrais..
Sucesso, Copiado para: /content/dados_locais/base_dados_cadastrais
Procecando pasta base_score_bureau_movel..
Sucesso, Copiado para: /content/dados_locais/base_score_bureau_movel
Procecando pasta base_score_bureau_movel_full..
Sucesso, Copiado para: /content/dados_locais/base_score_bureau_movel_full
Procecando pasta base_telco..
Sucesso, Copiado para: /content/dados_locais/base_telco
Procecando pasta book_pagamento/dados_pagamento..
Sucesso, Copiado para: /content/dados_locais/book_pagamento/dados_pagamento
Procecando pasta book_pagamento..
Sucesso, Copiado para: /content/dados_locais/book_pagamento
Procecando pasta bases_recarga..
Sucesso, Copiado para: /content/dados_locais/bases_recarga

--PROCESSO FINALIZADO--
Pastas criadas localmente: ['base_score_bureau_movel_full', 'base_telco', 'book_pagamento', 'bases_recarga', 'base_score_bureau_movel', 'base_dados_cadastrais']


In [9]:
# --------------------------------------------
# Carregando os dados para um unico dicionario
# --------------------------------------------

dfs = {}
path_base = "/content/dados_locais"

print('--- Carregando Bases Parquet (Fatos) ---')

# 1. Base cadastral
dfs['cadastral'] = spark.read.parquet(f'{path_base}/base_dados_cadastrais')

# 2. Score Bureau Full
dfs['score_bureau_full'] = spark.read.parquet(f'{path_base}/base_score_bureau_movel_full/*.parquet')

# 3. Telco
dfs['telco'] = spark.read.parquet(f'{path_base}/base_telco/*.parquet')

# 4. Pagamento
dfs['pagamento'] = spark.read.parquet(f"{path_base}/book_pagamento/dados_pagamento/*.parquet")

# 5. Recarga
dfs['recarga'] = spark.read.parquet(f'{path_base}/bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/*.parquet')

print('Carregado bases Parquet:')
print(dfs.keys())

# 6. Arquivos BI_DIM .csv
print('\n--- Carregando CSVs ---')
try:
  path_recarga = f'{path_base}/bases_recarga'

  csvs = [f for f in os.listdir(path_recarga) if f.endswith('.csv')]

  for arquivo in csvs:
    nome_limpo = arquivo.replace('.csv', '').lower()

    dfs[nome_limpo] = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{path_recarga}/{arquivo}')
    print(f'Carregado CSV: {nome_limpo}')
except Exception as e:
  print(f'Não foi possivel carregar: {e}')

--- Carregando Bases Parquet (Fatos) ---
Carregado bases Parquet:
dict_keys(['cadastral', 'score_bureau_full', 'telco', 'pagamento', 'recarga'])

--- Carregando CSVs ---
Carregado CSV: bi_dim_plataforma
Carregado CSV: bi_dim_tipo_insercao
Carregado CSV: bi_dim_tipo_recarga
Carregado CSV: bi_dim_status_plataforma
Carregado CSV: bi_dim_forma_pagamento
Carregado CSV: bi_dim_tecnologia
Carregado CSV: bi_dim_instituicao
Carregado CSV: bi_dim_plano_preco
Carregado CSV: bi_dim_promocao_credito
Carregado CSV: bi_dim_tipo_credito
Carregado CSV: bi_dim_canal_aquisicao_credito


#

# 3 - Criação do ID unico e identificação do grupo controle

## 3.1 - Tabelas Com NUM_CPF e SAFRA

In [25]:
#----------------------------------------------
# ID_UNICO -> NUM_CPF + SAFRA; e GRUPO_CONTROLE
#----------------------------------------------
tabelas_com_safra = ['cadastral', 'score_bureau_full', 'telco']

for tabela in tabelas_com_safra:

  # Criando ID_UNICO com NUM_CPF + SAFRA
  dfs[tabela] = dfs[tabela].withColumn(
      'ID_UNICO',
      concat(col('NUM_CPF'), col('SAFRA').cast('string'))
  )

  # Criando Flag do grupo controle GRUPO_CONTROLE
  # O grupo controle pode ser identificado a partir do 6º e 7º dígitos do CPF,
  # considerando as combinações ZZ e ZX.
  dfs[tabela] = dfs[tabela].withColumn(
      'GRUPO_CONTROLE',
      substring(col('NUM_CPF'), 6, 2).isin(['ZZ', 'ZX'])
  )

## 3.2 - Tabelas com NUM_CPF e sem SAFRA

In [26]:
#-----------------------------------
# Criação somente do GRUPO_CONTROLE
#-----------------------------------

tabelas_sem_safra = ['pagamento', 'recarga']

for tabela in tabelas_sem_safra:

  dfs[tabela] = dfs[tabela].withColumn(
      'GRUPO_CONTROLE',
      substring(col('NUM_CPF'), 6, 2).isin(['ZZ', 'ZX'])
  )

#

# 4 - Tratamento de Valores Ausentes

## 4.1 - Tabela cadastral

In [ ]:
df = dfs['cadastral']

In [ ]:
df = df.withColumns({
    'SALARIO_FUNC_PUBL': F.coalesce(F.col('var_11').cast('double'), F.lit(-1.0)),
    'SALARIO_FUNC_PUBL_MISSING': F.when(F.col('var_11').isNotNull(), 0).otherwise(1),
    'CARGO_FUNC_PUBL': F.coalesce(F.col('var_10'), F.lit('NAO_APLICAVEL')),
    'FUNC_PUBL': F.when(F.col('var_20') == 'FUNC_PUBL', 1).otherwise(0),

    'var_02_missing': F.when(F.col('var_02').isNotNull(), 0).otherwise(1),
    'var_02' : F.coalesce(F.col('var_02').cast('double'), F.lit(-999)),

    'VALOR_EMPR_DIRETOR': F.coalesce(F.col('var_14').cast('float'), F.lit(-999)),
    'VALOR_EMPR_DIRETOR_MISSING': F.when(F.col('var_14').isNotNull(), 0).otherwise(1),
    'EMPR_DIRETOR': F.when(F.col('var_22').isNotNull(), 1).otherwise(0),
    'BOLSA_FAMILIA': F.when(F.col('var_23').isNotNull(), 1).otherwise(0),

    'SAFRA_BOLSA_FAMILIA': F.coalesce(F.col('var_17').cast('int'), F.lit(190001)),
    'SAFRA_BOLSA_FAMILIA_MISSING': F.when(F.col('var_17').isNotNull(), 0).otherwise(1),
    'UF_BOLSA_FAMILIA': F.coalesce(F.col('var_15'), F.lit('NAO_APLICAVEL')),
    'BENEFICIO_BOLSA_FAMILIA': F.coalesce(F.col('var_16').cast('float'), F.lit(0)),
    'BENEFICIO_BOLSA_FAMILIA_MISSING': F.when(F.col('var_16').isNotNull(), 0).otherwise(1),

    'var_07_MONETARIO': F.coalesce(F.col('var_07').cast('float'), F.lit(-999)),
    'var_07_MONETARIO_MISSING': F.when(F.col('var_07').isNotNull(), 0).otherwise(1),

    'NUMERO_APOSENTADO': F.coalesce(F.col('var_08').cast('int'), F.lit(-999)),
    'NUMERO_APOSENTADO_MISSING': F.when(F.col('var_08').isNotNull(), 0).otherwise(1),
    'APOSENTADO': F.when(F.col('var_18').isNotNull(), 1).otherwise(0),

    'AUXILLIO_EMERGENCIAL': F.when(F.col('var_19').isNotNull(), 1).otherwise(0),
    'MESES_AUXILIO_EMERGENCIAL': F.coalesce(F.col('var_09').cast('int'), F.lit(0)),
    'MESES_AUXILIO_EMERGENCIAL_MISSING': F.when(F.col('var_09').isNotNull(), 0).otherwise(1),

    'DATA_FUNC_PRIVADO': F.coalesce(F.to_date(F.col('var_12' ), 'dd/MM/yyyy'), F.lit('9999-01-01').cast('date')),
    'DATA_FUNC_PRIVADO_MISSING': F.when(F.col('var_12').isNotNull(), 0).otherwise(1),
    'FUNC_PRIVADO': F.when(F.col('var_21').isNotNull(), 1).otherwise(0),
    'STATUS_FUNC_PRIVADO': F.coalesce(F.col('var_24'), F.lit('NAO_APLICAVEL')),

    'flag_mig2': F.coalesce(F.col('flag_mig2'), F.lit('SEM_MIGRACAO')),

    'CONSOLIDADO': F.coalesce(F.col('var_25'), F.lit('SEM_INFORMACAO')),

    'CEP_3_digitos': F.coalesce(F.col('CEP_3_digitos'), F.lit('XXX')),

    'var_03': F.coalesce(F.col('var_03').cast('int'), F.lit(-1)),
    'var_03_MISSING': F.when(F.col('var_03') == -1, 1).otherwise(0),

    'var_05': F.coalesce(F.col('var_05').cast('int'), F.lit(-1)),
    'var_05_MISSING': F.when(F.col('var_05') == -1, 1).otherwise(0),

    'DATADENASCIMENTO': F.coalesce(F.to_date(F.col('DATADENASCIMENTO'), 'dd/MM/yyyy'), F.lit('1000-01-01').cast('date')),
    'DATADENASCIMENTO_MISSING': F.when(F.col('DATADENASCIMENTO') == '1000-01-01', 1).otherwise(0),

    'STATUSRF': F.coalesce(F.col('STATUSRF'), F.lit('SEM_STATUS')),
    'NUM_STATUSRF': F.coalesce(F.col('var_04').cast('int'), F.lit(-1))

}).drop('var_11', 'var_10', 'var_20', 'var_14', 'var_22', 'var_23', 'var_17', 'var_15', 'var_16',
        'var_07', 'var_13', 'var_08', 'var_18', 'var_06', 'var_19', 'var_09', 'var_12', 'var_21',
        'var_24','var_25', 'var_04')

In [ ]:
ordem = ['ID_UNICO','GRUPO_CONTROLE','NUM_CPF','SAFRA','PROD', 'FLAG_INSTALACAO', 'flag_mig2','FPD', 'STATUSRF', 'NUM_STATUSRF',
         'DATADENASCIMENTO', 'DATADENASCIMENTO_MISSING','FUNC_PUBL', 'CARGO_FUNC_PUBL', 'SALARIO_FUNC_PUBL', 'SALARIO_FUNC_PUBL_MISSING',
         'EMPR_DIRETOR', 'VALOR_EMPR_DIRETOR', 'VALOR_EMPR_DIRETOR_MISSING', 'BOLSA_FAMILIA', 'SAFRA_BOLSA_FAMILIA', 'SAFRA_BOLSA_FAMILIA_MISSING',
         'UF_BOLSA_FAMILIA', 'BENEFICIO_BOLSA_FAMILIA', 'BENEFICIO_BOLSA_FAMILIA_MISSING', 'APOSENTADO', 'NUMERO_APOSENTADO', 'NUMERO_APOSENTADO_MISSING',
         'AUXILLIO_EMERGENCIAL', 'MESES_AUXILIO_EMERGENCIAL', 'MESES_AUXILIO_EMERGENCIAL_MISSING','FUNC_PRIVADO' , 'STATUS_FUNC_PRIVADO',
         'DATA_FUNC_PRIVADO', 'DATA_FUNC_PRIVADO_MISSING', 'CONSOLIDADO', 'CEP_3_digitos', 'var_07_MONETARIO', 'var_07_MONETARIO_MISSING',
         'var_02', 'var_02_missing', 'var_03', 'var_03_MISSING','var_05', 'var_05_MISSING']

In [ ]:
df_save = df[ordem]

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/Cadastral/tabela_cadastral"
df_save.write.mode("overwrite").parquet(path_silver)

## 4.2 Tabela Telco

In [ ]:
df = dfs['telco']

In [ ]:
df = df.withColumns({

    'flag_mig2': F.coalesce(F.col('flag_mig2'), F.lit('SEM_MIGRACAO')),

    'var26_a_var77_MISSING': F.when(F.col("var_26").isNull(), 1).otherwise(0)

})

In [ ]:
numericas_nulos = ['var_26', 'var_27', 'var_28', 'var_29', 'var_30', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38',
 'var_39', 'var_40', 'var_41', 'var_42', 'var_43', 'var_44', 'var_45', 'var_46', 'var_47', 'var_48', 'var_49', 'var_50', 'var_51', 'var_52',
 'var_53', 'var_54', 'var_55', 'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66',
 'var_67', 'var_68', 'var_69', 'var_70', 'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77']

In [ ]:
for col in numericas_nulos:
    df = df.withColumn(
        col,
        F.coalesce(F.col(col).cast('decimal(18,2)'), F.lit(-999))
    )

In [ ]:
nao_nulas = ['var_78', 'var_79', 'var_80','var_81', 'var_82', 'var_83', 'var_84', 'var_85', 'var_86', 'var_87', 'var_88', 'var_89',
             'var_90', 'var_91', 'var_92', 'var_93']

In [ ]:
for col in nao_nulas:
    df = df.withColumn(
        col,
        F.coalesce(F.col(col).cast('decimal(18,2)'), F.lit(-999))
    )

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_telco"
df.write.mode("overwrite").parquet(path_silver)

## 4.3 Tabela score_bureau_full

In [ ]:
df = dfs['score_bureau_full']

In [ ]:
df = df.withColumns({
    'SCORE_01_MISSING': F.when(F.col('SCORE_01').isNull(), 1).otherwise(0),
    'SCORE_01': F.coalesce(F.col('SCORE_01').cast('int'), F.lit(-1)),

    'SCORE_02_MISSING': F.when(F.col('SCORE_02').isNull(), 1).otherwise(0),
    'SCORE_02': F.coalesce(F.col('SCORE_02').cast('int'), F.lit(-1))
})

In [ ]:
df = df.select('ID_UNICO', 'GRUPO_CONTROLE', 'NUM_CPF', 'SAFRA', 'FLAG_INSTALACAO', 'PROD','FPD', 'flag_mig2',
               'SCORE_01', 'SCORE_01_MISSING', 'SCORE_02', 'SCORE_02_MISSING')

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_score_bureau_full"
df.write.mode("overwrite").parquet(path_silver)

## 4.4 - Tabelas Dimensão

### 4.4.1 - bi_dim_tipo_insercao

In [ ]:
df = dfs['bi_dim_tipo_insercao']

In [ ]:
df = df.withColumns({
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_EXPIRACAO_DW': F.coalesce(F.col('DAT_EXPIRACAO_DW'), F.lit("9999-12-31").cast("date"))
})

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_tipo_insercao"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.2 - bi_dim_promocao_credito

In [ ]:
df = dfs['bi_dim_promocao_credito']

In [ ]:
df = df.withColumns({

    'DAT_EXPIRACAO_DW': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),

    'DAT_ATUALIZACAO_DW': F.to_date(F.col('DAT_ATUALIZACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_INICIO_VIGENCIA': F.to_date(F.col('DAT_INICIO_VIGENCIA'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_FIM_VIGENCIA': F.to_date(F.col('DAT_FIM_VIGENCIA'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
registro = [(-1, 'Não se aplica', date(9999,12,31), date(1900, 1, 1),date(1900, 1, 1), -1, 'Não se aplica', 'Não se aplica', date(1900, 1, 1), date(9999,12,31), -1.0, -1)]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|COD_PROMOCAO|        DSC_PROMOCAO|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|COD_PROM_GRUPO_CARTAO|   DSC_NOME_PROMOCAO|COD_TIPO_PROMOCAO|DAT_INICIO_VIGENCIA|DAT_FIM_VIGENCIA|VAL_PROMOCAO|NUM_CONTA_DEDICADA|
+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|         317|Incentivo a adesã...|      9999-12-31|        2025-09-14|    2011-11-11|                    1|            DEBAU_01|  INCENTIVO_DEBAU|         2007-04-07|      2010-04-30|        50.0|                 2|
|         318|Incentivo à recar...|      9999-12-31|        2025-09-14|    2011-11-11|                    2|   Recarga Turbinada|   

In [ ]:
registro = [(-4, 'DESCONHECIDO', date(9999,12,31), date(1900, 1, 1),date(1900, 1, 1), -4, 'DESCONHECIDO', 'DESCONHECIDO', date(1900, 1, 1), date(9999,12,31), -4.0, -4)]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|COD_PROMOCAO|        DSC_PROMOCAO|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|COD_PROM_GRUPO_CARTAO|   DSC_NOME_PROMOCAO|COD_TIPO_PROMOCAO|DAT_INICIO_VIGENCIA|DAT_FIM_VIGENCIA|VAL_PROMOCAO|NUM_CONTA_DEDICADA|
+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|         317|Incentivo a adesã...|      9999-12-31|        2025-09-14|    2011-11-11|                    1|            DEBAU_01|  INCENTIVO_DEBAU|         2007-04-07|      2010-04-30|        50.0|                 2|
|         318|Incentivo à recar...|      9999-12-31|        2025-09-14|    2011-11-11|                    2|   Recarga Turbinada|   

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_promocao_credito"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.3 - bi_dim_promocao_credito

In [ ]:
df = dfs['bi_dim_promocao_credito']

In [ ]:
df = df.withColumns({

    'DAT_EXPIRACAO_DW' : F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),

    'DAT_ATUALIZACAO_DW': F.to_date(F.col('DAT_ATUALIZACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_INICIO_VIGENCIA': F.to_date(F.col('DAT_INICIO_VIGENCIA'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_FIM_VIGENCIA': F.to_date(F.col('DAT_FIM_VIGENCIA'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
registro = [(-1, 'Não se aplica', date(9999,12,31), date(1900, 1, 1),date(1900, 1, 1), -1, 'Não se aplica', 'Não se aplica', date(1900, 1, 1), date(9999,12,31), -1.0, -1)]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|COD_PROMOCAO|        DSC_PROMOCAO|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|COD_PROM_GRUPO_CARTAO|   DSC_NOME_PROMOCAO|COD_TIPO_PROMOCAO|DAT_INICIO_VIGENCIA|DAT_FIM_VIGENCIA|VAL_PROMOCAO|NUM_CONTA_DEDICADA|
+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|         317|Incentivo a adesã...|      9999-12-31|        2025-09-14|    2011-11-11|                    1|            DEBAU_01|  INCENTIVO_DEBAU|         2007-04-07|      2010-04-30|        50.0|                 2|
|         318|Incentivo à recar...|      9999-12-31|        2025-09-14|    2011-11-11|                    2|   Recarga Turbinada|   

In [ ]:
registro = [(-4, 'DESCONHECIDO', date(9999,12,31), date(1900, 1, 1),date(1900, 1, 1), -4, 'DESCONHECIDO', 'DESCONHECIDO', date(1900, 1, 1), date(9999,12,31), -4.0, -4)]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|COD_PROMOCAO|        DSC_PROMOCAO|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|COD_PROM_GRUPO_CARTAO|   DSC_NOME_PROMOCAO|COD_TIPO_PROMOCAO|DAT_INICIO_VIGENCIA|DAT_FIM_VIGENCIA|VAL_PROMOCAO|NUM_CONTA_DEDICADA|
+------------+--------------------+----------------+------------------+--------------+---------------------+--------------------+-----------------+-------------------+----------------+------------+------------------+
|         317|Incentivo a adesã...|      9999-12-31|        2025-09-14|    2011-11-11|                    1|            DEBAU_01|  INCENTIVO_DEBAU|         2007-04-07|      2010-04-30|        50.0|                 2|
|         318|Incentivo à recar...|      9999-12-31|        2025-09-14|    2011-11-11|                    2|   Recarga Turbinada|   

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_promocao_credito"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.4 - bi_dim_canal_aquisicao_credito

In [ ]:
df = dfs['bi_dim_canal_aquisicao_credito']

In [ ]:
df = df.withColumns({
    'DAT_EXPIRACAO_DW': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),

    'COD_TIPO_CREDITO': F.coalesce(F.col('COD_TIPO_CREDITO'), F.lit('NAO INFORMADO')),
    'COD_AGENTE_CREDITO': F.coalesce(F.col('COD_AGENTE_CREDITO'), F.lit('NAO INFORMADO')),

    'COD_TIPO_INSTITUICAO': F.coalesce(F.col('COD_TIPO_INSTITUICAO'), F.lit(-4)),
    'DSC_TIPO_INSTITUICAO': F.coalesce(F.col('DSC_TIPO_INSTITUICAO'), F.lit('DESCONHECIDO')),

    'DAT_ATUALIZACAO_DW': F.to_date(F.col('DAT_ATUALIZACAO_DW'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_canal_aquisicao_credito"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.5 - bi_dim_status_plataforma

In [ ]:
df = dfs['bi_dim_status_plataforma']

In [ ]:
df = df.withColumns({
    'DAT_ATUALIZACAO_DW': F.to_date(F.col('DAT_ATUALIZACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_bi_dim_status_plataforma"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.6 - bi_dim_tipo_recarga

In [ ]:
df = dfs['bi_dim_tipo_recarga']

In [ ]:
df = df.withColumns({
    'DAT_EXPIRACAO_DW': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
registro = [(-4, 'DESCONHECIDO', date(9999,12,31), date(1900, 1, 1))]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+---------------+--------------------+----------------+--------------+
|DW_TIPO_RECARGA|    DSC_TIPO_RECARGA|DAT_EXPIRACAO_DW|DAT_CRIACAO_DW|
+---------------+--------------------+----------------+--------------+
|             -3|       Não Informado|      9999-12-31|    2011-10-19|
|             -2|     Não Determinado|      9999-12-31|    2011-10-19|
|             -1|       Não se Aplica|      9999-12-31|    2011-10-19|
|              1|Franquia do Claro...|      9999-12-31|    2011-10-19|
|              2|Adicional do Clar...|      9999-12-31|    2011-10-19|
|             -4|        DESCONHECIDO|      9999-12-31|    1900-01-01|
+---------------+--------------------+----------------+--------------+



In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_tipo_recarga"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.7 - bi_dim_plano_preco

In [ ]:
df = dfs['bi_dim_plano_preco']

In [ ]:
df = df.withColumns({
    'DAT_EXPIRACAO_DW': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),
    'DAT_EFETIVACAO': F.coalesce(F.to_date(F.col('DAT_EFETIVACAO'), "ddMMMyyyy:HH:mm:ss"), F.lit("1900-01-01").cast("date")),
    'DAT_EXPIRACAO': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),
    'DSC_PLANO_PRECO_UNICO_BI': F.coalesce(F.col('DSC_PLANO_PRECO_UNICO_BI'),F.lit('-4')),
    'COD_PLANO_COMPONENTE': F.coalesce(F.col('COD_PLANO_COMPONENTE'), F.lit(-4)),
    'NUM_FRANQUIA_VOLUME_BI': F.coalesce(F.col('NUM_FRANQUIA_VOLUME_BI').cast('int'), F.lit(-4)),
    'DSC_MODALIDADE_PLANO': F.coalesce(F.col('DSC_MODALIDADE_PLANO'), F.lit('DESCONHECIDO')),
    'NUM_FRANQUIA_REAIS_BI': F.coalesce(F.col('NUM_FRANQUIA_REAIS_BI').cast('int'), F.lit(-4)),
    'NUM_FRANQUIA_EVENTOS_BI': F.coalesce(F.col('NUM_FRANQUIA_EVENTOS_BI').cast('int'), F.lit(-4)),
    'NUM_FRANQUIA_MINUTOS_BI': F.coalesce(F.col('NUM_FRANQUIA_MINUTOS_BI').cast('int'), F.lit(-4)),
    'DAT_ATUALIZACAO_DW': F.to_date(F.col('DAT_ATUALIZACAO_DW'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
registro = [(-4, '-4', '-4','-4','-4', 4, date(1900,1,1), date(9999,12,31),'-4', '-4', '-4', 'DESC', '-4',  -4, '-4', date(9999,12,31),date(1900,1,1), date(1900,1,1), -4, -4, -4, -4, -4, '-4', 'DESCONHECIDO')]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+--------+---------------+--------------------+----------------+--------------------+---------------+--------------+-------------+--------------------+------------------+-----------------+-------------------+-----------------------+--------------+-----------------+----------------+------------------+--------------+-----------------------+---------------------+-----------------------+----------------------+--------------------+------------------------+--------------------+
|DW_PLANO|COD_PLANO_PRECO|     DSC_PLANO_PRECO|COD_TIPO_CLIENTE|COD_SUB_TIPO_CLIENTE|DW_TIPO_CLIENTE|DAT_EFETIVACAO|DAT_EXPIRACAO|  DSC_PLANO_PRECO_BI|DSC_GRUPO_PLANO_BI|DSC_TIPO_PLANO_BI|IND_AMDOCS_PLAT_PRE|COD_TRATAMENTO_ESPECIAL|COD_SISTEMA_DW|COD_TECNOLOGIA_DW|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|NUM_FRANQUIA_MINUTOS_BI|NUM_FRANQUIA_REAIS_BI|NUM_FRANQUIA_EVENTOS_BI|NUM_FRANQUIA_VOLUME_BI|COD_PLANO_COMPONENTE|DSC_PLANO_PRECO_UNICO_BI|DSC_MODALIDADE_PLANO|
+--------+---------------+--------------------

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_plano_preco"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.8 - bi_dim_forma_pagamento

In [ ]:
df = dfs['bi_dim_forma_pagamento']

In [ ]:
df = df.withColumns({
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

    'DAT_EXPIRACAO_DW': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date"))
})

In [ ]:
registro = [(-4, '-4', 'DESCONHECIDO', date(1900,1,1), date(9999,12,31))]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+------------------+-------------------+--------------------+--------------+----------------+
|DW_FORMA_PAGAMENTO|COD_FORMA_PAGAMENTO| DSC_FORMA_PAGAMENTO|DAT_CRIACAO_DW|DAT_EXPIRACAO_DW|
+------------------+-------------------+--------------------+--------------+----------------+
|                -1|                 -1|       Não se Aplica|    2008-02-21|      9999-12-31|
|                -2|                 -2|     Não Determinado|    2008-02-21|      9999-12-31|
|                -3|                 -3|       Não Informado|    2008-02-21|      9999-12-31|
|                11|                 CP|  Credito de PrePago|    2008-02-15|      9999-12-31|
|                12|                 DD|       Debito Direto|    2008-02-15|      9999-12-31|
|                13|                 MP|    Pagamento Manual|    2008-02-15|      9999-12-31|
|                14|                 PB|Arrecadacao Bancaria|    2008-02-15|      9999-12-31|
|                10|                 CA|    Pagamento Online

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_forma_pagamento"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.9 - bi_dim_instituicao

In [ ]:
df = dfs['bi_dim_instituicao']

In [ ]:
df = df.withColumns({
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_EXPIRACAO_DW': F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date"))
})

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_instituicao"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.10 - bi_dim_plataforma

In [ ]:
df = dfs['bi_dim_plataforma']

In [ ]:
df = df.withColumns({
    'DAT_EXPIRACAO_DW' : F.coalesce(F.to_date(F.col('DAT_EXPIRACAO_DW'), "ddMMMyyyy:HH:mm:ss"), F.lit("9999-12-31").cast("date")),
    'DAT_ATUALIZACAO_DW' : F.to_date(F.col('DAT_ATUALIZACAO_DW'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),
    'DSC_GRUPO_PLATAFORMA' : F.coalesce(F.col('DSC_GRUPO_PLATAFORMA'), F.lit('Outros'))
})

In [ ]:
registro = [(-4, '-4', date(9999,12,31), date(1900,1,1), date(1900,1,1), 'DESCONHECIDO', 'DESCONHECIDO', 'Outros')]
df_extra = spark.createDataFrame(registro, schema = df.schema)

df = df.union(df_extra)

+--------------+--------------+----------------+------------------+--------------+--------------------+-----------------------+--------------------+
|COD_PLATAFORMA|DSC_PLATAFORMA|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|   DSC_PLATAFORMA_BI|COD_GRUPO_PLATAFORMA_BI|DSC_GRUPO_PLATAFORMA|
+--------------+--------------+----------------+------------------+--------------+--------------------+-----------------------+--------------------+
|            24|          M2MS|      9999-12-31|        2018-11-06|    2018-11-06|Machine to Machin...|                  POSTL|          Telemetria|
|            26|         MVNOD|      9999-12-31|        2021-12-06|    2021-12-06|Parceiros MVNO DI...|                   MVNO|        MVNO Digital|
|            12|         POSTL|      9999-12-31|        2012-11-07|    2012-11-07|          Telemetria|                  POSTL|          Telemetria|
|            20|         POSDT|      9999-12-31|        2017-12-13|    2017-12-13|Pós Pago Deutsche...|   

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_plataforma"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.11 - bi_dim_tecnologia

In [ ]:
df = dfs['bi_dim_tecnologia']

In [ ]:
mask = "ddMMMyyyy:HH:mm:ss"
df = df.withColumns({
    'DAT_ATUALIZACAO_DW': F.coalesce(F.to_date(F.col('DAT_ATUALIZACAO_DW'), mask),
                                     F.to_date(F.lit('07MAY2010:11:23:38'), mask)),

    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), mask),
})

In [ ]:
df = df.withColumns({
    'COD_TECNOLOGIA_SVA':  F.when(F.col('COD_TECNOLOGIA_DW') == '-1', 'Não se aplica').otherwise(F.col('COD_TECNOLOGIA_SVA')),
    'COD_TECNOLOGIA_SVA': F.when(F.col('COD_TECNOLOGIA_DW') == '-2', 'Não definido').otherwise(F.col('COD_TECNOLOGIA_SVA')),
    'COD_TECNOLOGIA_SVA': F.when(F.col('COD_TECNOLOGIA_DW') == '-3', 'Não informado').otherwise(F.col('COD_TECNOLOGIA_SVA')),
    'COD_TECNOLOGIA_SVA': F.coalesce(F.col('COD_TECNOLOGIA_SVA'), F.lit('DESCONHECIDO'))
})

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_tecnologia"
df.write.mode("overwrite").parquet(path_silver)

### 4.4.12 - bi_dim_tipo_credito

In [ ]:
df = dfs['bi_dim_tipo_credito']

In [ ]:
df.show()

+----------------+--------------------+----------------+------------------+------------------+
|COD_TIPO_CREDITO|    DSC_TIPO_CREDITO|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|    DAT_CRIACAO_DW|
+----------------+--------------------+----------------+------------------+------------------+
|              PE|              Online|            NULL|              NULL|19OCT2011:14:39:55|
|              -1|                  NA|            NULL|01MAR2004:19:54:06|01MAR2004:19:54:06|
|              -2|                  ND|            NULL|01MAR2004:19:54:08|01MAR2004:19:54:08|
|              -3|                  NI|            NULL|01MAR2004:19:54:09|01MAR2004:19:54:09|
|              AU|        Autocontrole|            NULL|01MAR2004:18:18:22|01MAR2004:18:18:22|
|              BO|               Bonus|            NULL|01MAR2004:18:18:32|01MAR2004:18:18:32|
|              CF|       Cartão Físico|            NULL|01MAR2004:18:18:48|01MAR2004:18:18:48|
|              CV|      Pin Eletronico|           

In [ ]:
mask = "ddMMMyyyy:HH:mm:ss"
df = df.withColumns({
    'DAT_ATUALIZACAO_DW': F.coalesce(F.to_date(F.col('DAT_ATUALIZACAO_DW'), mask),
                                     F.to_date(F.lit('19OCT2011:14:39:55'), mask)),

    'DAT_CRIACAO_DW': F.to_date(F.col('DAT_CRIACAO_DW'), mask),

    'DAT_EXPIRACAO_DW' : F.coalesce(F.col('DAT_EXPIRACAO_DW'), F.lit('9999-12-31'))

})

In [ ]:
registro = [('-4', 'DESC', '9999-12-31', '1900-01-01', '1900-01-01')]

colunas = ["COD_TIPO_CREDITO", "DSC_TIPO_CREDITO", "DAT_EXPIRACAO_DW", "DAT_ATUALIZACAO_DW", "DAT_CRIACAO_DW"]
df_extra = spark.createDataFrame(registro, colunas)

df_extra = df_extra.withColumn("DAT_EXPIRACAO_DW", F.col("DAT_EXPIRACAO_DW").cast("date")) \
                   .withColumn("DAT_ATUALIZACAO_DW", F.col("DAT_ATUALIZACAO_DW").cast("date")) \
                   .withColumn("DAT_CRIACAO_DW", F.col("DAT_CRIACAO_DW").cast("date"))

df = df.union(df_extra)

+----------------+--------------------+----------------+------------------+--------------+
|COD_TIPO_CREDITO|    DSC_TIPO_CREDITO|DAT_EXPIRACAO_DW|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|
+----------------+--------------------+----------------+------------------+--------------+
|              PE|              Online|      9999-12-31|        2011-10-19|    2011-10-19|
|              -1|                  NA|      9999-12-31|        2004-03-01|    2004-03-01|
|              -2|                  ND|      9999-12-31|        2004-03-01|    2004-03-01|
|              -3|                  NI|      9999-12-31|        2004-03-01|    2004-03-01|
|              AU|        Autocontrole|      9999-12-31|        2004-03-01|    2004-03-01|
|              BO|               Bonus|      9999-12-31|        2004-03-01|    2004-03-01|
|              CF|       Cartão Físico|      9999-12-31|        2004-03-01|    2004-03-01|
|              CV|      Pin Eletronico|      9999-12-31|        2004-03-01|    2004-03-01|

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_bi_dim_tipo_credito"
df.write.mode("overwrite").parquet(path_silver)

## 4.5 - Tabela recarga

In [ ]:
df = dfs['recarga']

In [ ]:
df = df.withColumns({
    'VALOR_SOS': F.coalesce(F.col('VALOR_SOS').cast('float'), F.lit(0)),
    'DW_TIPO_RECARGA': F.coalesce(F.col('DW_TIPO_RECARGA').cast('int'), F.lit(-4)),
    'DW_PLANO_TARIFACAO': F.coalesce(F.col('DW_PLANO_TARIFACAO').cast('int'), F.lit(-4)),
    'COD_PROMOCAO' : F.coalesce(F.col('COD_PROMOCAO').cast('int'), F.lit(-4)),
    'DAT_INSERCAO_CREDITO': F.to_date(F.col('DAT_INSERCAO_CREDITO'), "ddMMMyyyy:HH:mm:ss")
})

In [ ]:
valor_bigint = ['DW_NUM_NTC',  'DW_NUM_CLIENTE',]
valor_int = ['COD_CANAL_AQUISICAO', 'HOR_INSERCAO_CREDITO','DW_TIPO_INSERCAO','DW_FORMA_PAGAMENTO','DW_INSTITUICAO', 'FLAG_SOS']
valor_float = ['VAL_CREDITO_INSERIDO', 'VAL_BONUS', 'VAL_REAl', ]

In [ ]:
for coluna in valor_bigint:
  df = df.withColumn(
      coluna,
      F.col(coluna).cast('bigint')
  )

In [ ]:
for coluna in valor_int:
  df = df.withColumn(
      coluna,
      F.col(coluna).cast('int')
  )

In [ ]:
for coluna in valor_float:
  df = df.withColumn(
      coluna,
      F.col(coluna).cast('float')
  )

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_recarga"
df.write.mode("overwrite").parquet(path_silver)

## 4.6 Tabela pagamento

In [10]:
df = dfs['pagamento']

In [12]:
NULOS_100 = ['COD_NETUNO_PAGAMENTO', 'COD_DESALOCACAO_CREDITO', 'DAT_ATUALIZACAO_CREDITO']

In [13]:
for coluna in NULOS_100:
  df = df.withColumn(
      coluna,
      F.coalesce(F.col(coluna), F.lit('DESCONHECIDO'))
  )

In [15]:
df = df.withColumns({
    'COD_FUNDO_ATIVIDADE': F.coalesce(F.col('COD_FUNDO_ATIVIDADE').cast('int'), F.lit(-4)),
    'COD_FUNDO_ATIVIDADE_MISSING': F.when(F.col('COD_FUNDO_ATIVIDADE') == -4, 1).otherwise(0),

    'NUM_PARCELA_PAGAMENTO': F.coalesce(F.col('NUM_PARCELA_PAGAMENTO').cast('int'), F.lit(-4)),
    'NUM_PARCELA_PAGAMENTO_MISSING': F.when(F.col('NUM_PARCELA_PAGAMENTO') == -4, 1).otherwise(0),

    'DAT_ATUALIZACAO_ATIVIDADE': F.coalesce(F.to_date(F.col('DAT_ATUALIZACAO_ATIVIDADE'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),
    'DAT_ATUALIZACAO_ATIVIDADE_MISSING' : F.when(F.col('DAT_ATUALIZACAO_ATIVIDADE') == '1900-01-01', 1).otherwise(0),

    'NUM_CONTA_ATIVIDADE': F.coalesce(F.col('NUM_CONTA_ATIVIDADE'), F.lit('-4')),

    'DAT_ATUALIZACAO_PAGAMENTO_MISSING': F.when(F.col('DAT_ATUALIZACAO_PAGAMENTO').isNull(), 1).otherwise(0),
    'DAT_ATUALIZACAO_PAGAMENTO' : F.coalesce(F.to_date(F.col('DAT_ATUALIZACAO_PAGAMENTO'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),

    'DSC_PAGAMENTO': F.coalesce(F.col('DSC_PAGAMENTO'), F.lit('DESCONHECIDO')),

    'COD_ARQUIVO_PAGAMENTO': F.coalesce(F.col('COD_ARQUIVO_PAGAMENTO'), F.lit('DESCONHECIDO')),

    'COD_METODO_PAGAMENTO': F.coalesce(F.col('COD_METODO_PAGAMENTO').cast('int'), F.lit(-4)),

    'NUM_AGRUPADOR_MISSING': F.when(F.col('NUM_AGRUPADOR_PAGAMENTO').isNull(), 1).otherwise(0),
    'NUM_AGRUPADOR_PAGAMENTO': F.coalesce(F.col('NUM_AGRUPADOR_PAGAMENTO').cast('int'), F.lit(-4)),

    'COD_AGENCIA_ATIVIDADE': F.coalesce(F.col('COD_AGENCIA_ATIVIDADE'), F.lit('-4')),

    'COD_LOGIN_PAGAMENTO_MISSING': F.when(F.col('COD_LOGIN_PAGAMENTO').isNull(), 1).otherwise(0),
    'COD_LOGIN_PAGAMENTO': F.coalesce(F.col('COD_LOGIN_PAGAMENTO').cast('int'), F.lit(-4)),

    'COD_LOGIN_CREDITO_MISSING': F.when(F.col('COD_LOGIN_CREDITO').isNull(), 1).otherwise(0),
    'COD_LOGIN_CREDITO': F.coalesce(F.col('COD_LOGIN_CREDITO').cast('int'), F.lit(-4)),

    'COD_LOGIN_OPERADOR_ATIVIDADE_MISSING': F.when(F.col('COD_LOGIN_OPERADOR_ATIVIDADE').isNull(), 1).otherwise(0),
    'COD_LOGIN_OPERADOR_ATIVIDADE': F.coalesce(F.col('COD_LOGIN_OPERADOR_ATIVIDADE').cast('int'), F.lit(-4)),

    'SEQ_ARQUIVO_PAGAMENTO_MISSING': F.when(F.col('SEQ_ARQUIVO_PAGAMENTO').isNull(), 1).otherwise(0),
    'SEQ_ARQUIVO_PAGAMENTO': F.coalesce(F.col('SEQ_ARQUIVO_PAGAMENTO').cast('int'), F.lit(-4)),

    'COD_BANCO_ATIVIDADE': F.coalesce(F.col('COD_BANCO_ATIVIDADE'), F.lit('DESCONHECIDO')),

    'COD_ORIGEM_NETUNO': F.coalesce(F.col('COD_ORIGEM_NETUNO'), F.lit('DESCONHECIDO')),

    'IND_STATUS_PAGAMENTO': F.coalesce(F.col('IND_STATUS_PAGAMENTO'), F.lit('-4')),

    'COD_RAZAO_ATIVIDADE': F.coalesce(F.col('COD_RAZAO_ATIVIDADE'), F.lit('DESCONHECIDO')),

    'NUM_FATURA_PAGAMENTO': F.coalesce(F.col('NUM_FATURA_PAGAMENTO'), F.lit('DESCONHECIDO')),

    'VAL_ATUAL_PAGAMENTO_MISSING': F.when(F.col('VAL_ATUAL_PAGAMENTO').isNull(), 1).otherwise(0),
    'VAL_ATUAL_PAGAMENTO': F.coalesce(F.col('VAL_ATUAL_PAGAMENTO').cast('float'), F.lit(-4.0)),

    'BLOCO_28-03-percet_MISSING': F.when(F.col('SEQ_FATURA_CREDITO').isNull(), 1).otherwise(0),

    'SEQ_FATURA_CREDITO': F.coalesce(F.col('SEQ_FATURA_CREDITO').cast('int'), F.lit(-4)),
    'SEQ_PAGAMENTO_CREDITO': F.coalesce(F.col('SEQ_PAGAMENTO_CREDITO').cast('int'), F.lit(-4)),
    'SEQ_ENTIDADE_PAGAMENTO': F.coalesce(F.col('SEQ_ENTIDADE_PAGAMENTO').cast('int'), F.lit(-4)),
    'SEQ_ENTIDADE_ATIVIDADE': F.coalesce(F.col('SEQ_ENTIDADE_ATIVIDADE').cast('int'), F.lit(-4)),
    'SEQ_ENTIDADE_CREDITO': F.coalesce(F.col('SEQ_ENTIDADE_CREDITO').cast('int'), F.lit(-4)),

    'VAL_PAGAMENTO_CREDITO': F.coalesce(F.col('VAL_PAGAMENTO_CREDITO').cast('float'), F.lit(-4.0)),
    'VAL_ORIGINAL_PAGAMENTO': F.coalesce(F.col('VAL_ORIGINAL_PAGAMENTO').cast('float'), F.lit(-4.0)),
    'VAL_BAIXA_ATIVIDADE': F.coalesce(F.col('VAL_BAIXA_ATIVIDADE').cast('float'), F.lit(-444444.0)),

    'DAT_STATUS_PAGAMENTO': F.coalesce(F.to_date(F.col('DAT_STATUS_PAGAMENTO'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),
    'DAT_CRIACAO_CREDITO': F.coalesce(F.to_date(F.col('DAT_CRIACAO_CREDITO'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),
    'DAT_CRIACAO_PAGAMENTO': F.coalesce(F.to_date(F.col('DAT_CRIACAO_PAGAMENTO'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),
    'DAT_ATIVIDADE_CREDITO': F.coalesce(F.to_date(F.col('DAT_ATIVIDADE_CREDITO'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),
    'DAT_BAIXA_ATIVIDADE': F.coalesce(F.to_date(F.col('DAT_BAIXA_ATIVIDADE'), "ddMMMyyyy:HH:mm:ss"), F.lit('9999-12-31').cast("date")),
    'DAT_DEPOSITO_ATIVIDADE': F.coalesce(F.to_date(F.col('DAT_DEPOSITO_ATIVIDADE'), "ddMMMyyyy:HH:mm:ss"), F.lit('9999-12-31').cast("date")),
    'DAT_CRIACAO_ATIVIDADE': F.coalesce(F.to_date(F.col('DAT_CRIACAO_ATIVIDADE'), "ddMMMyyyy:HH:mm:ss"), F.lit('1900-01-01').cast("date")),
    'DAT_VENCIMENTO_CREDITO' : F.coalesce(F.to_date(F.col('DAT_VENCIMENTO_CREDITO'), "ddMMMyyyy:HH:mm:ss"), F.lit('9999-12-31').cast("date")),

    'IND_TIPO_CREDITO': F.coalesce(F.col('IND_TIPO_CREDITO'), F.lit('DESCONHECIDO')),

    'DSC_NOME_BANCO_PAGAMENTO': F.coalesce(F.col('DSC_NOME_BANCO_PAGAMENTO'), F.lit('DESCONHECIDO')),

    'COD_TIPO_PAGAMENTO': F.coalesce(F.col('COD_TIPO_PAGAMENTO'), F.lit('DESCONHECIDO')),
    'COD_FORMA_PAGAMENTO': F.coalesce(F.col('COD_FORMA_PAGAMENTO'), F.lit('DESCONHECIDO')),
    'COD_TIPO_FATURA': F.coalesce(F.col('COD_TIPO_FATURA'), F.lit('DESCONHECIDO')),
    'COD_ALOCACAO_CREDITO': F.coalesce(F.col('COD_ALOCACAO_CREDITO'), F.lit('DESCONHECIDO')),
    'COD_CONTA_ATIVIDADE': F.coalesce(F.col('COD_CONTA_ATIVIDADE'), F.lit('DESCONHECIDO')),
    'COD_ATIVIDADE': F.coalesce(F.col('COD_ATIVIDADE'), F.lit('DESCONHECIDO'))
})

In [17]:
datas = ['DAT_STATUS_FATURA', 'DAT_CRIACAO_DW', 'DAT_CRIACAO_ATIVIDADE', 'DAT_ATUALIZACAO_ATIVIDADE', 'DAT_BAIXA_ATIVIDADE', 'DAT_DEPOSITO_ATIVIDADE', 'DAT_CRIACAO_PAGAMENTO', 'DAT_ATUALIZACAO_PAGAMENTO',
         'DAT_STATUS_PAGAMENTO', 'DAT_CRIACAO_CREDITO',  'DAT_ATIVIDADE_CREDITO','DAT_VENCIMENTO_CREDITO' ]

In [18]:
for coluna in datas:
  df = df.withColumn(
      coluna,
      F.to_date(F.col(coluna), "ddMMMyyyy:HH:mm:ss")
  )

In [22]:
valores = ['VAL_DESCONTO_ITEM','VAL_PAGAMENTO_ITEM', 'VAL_JUROS_MULTAS_ITEM', 'VAL_MULTA_EQUIP_ITEM', 'VAL_MULTA_EQUIP_TOTAL', 'VAL_MULTA_FID_ITEM' ]

In [23]:
for coluna in valores:
    df = df.withColumn(
        coluna,
        F.coalesce(F.col(coluna).cast('float'), F.lit(0.0))
    )

In [ ]:
#---------------------
# MODIFICAR BUCKET
#----------------------
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Silver/tabela_pagamento2"
df.write.mode("overwrite").parquet(path_silver)